In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
import time
from sklearn.base import clone

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import make_scorer, f1_score, recall_score, roc_auc_score

from ex_01_read_data import get_welding_data
from ex_03_feature_extraction import extract_features

d:\Dokumente\€ ---- BSc Informatik\Übungsblätter\Sem 5\aML\Code Ordner\projekt\applied-machine-learning-project\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
data_path = Path("data/Welding/data.csv")
result_path = Path("model_analysis/ex_05")
result_path.mkdir(parents=True, exist_ok=True)

data, labels, exp_ids = get_welding_data(data_path)

## Exercise 5.1: Classifier Selection

Chosen classification algorithms:

1. **Random Forest**, i.e. `RandomForestClassifier` from `scikit-learn`
    - Random Forest ensembles reduce the overfitting tendency of single decision trees and have a high resistance to noise by averaging multiple trees into a communal decision. Additionally they handle high-dimensional data well and are suitable for usage on features extraced from the time series data. As previous tasks have shown, the welding dataset contains complex patterns across multiple dimensions and contains a substantial amount of noise, complicating the process of detecting patterns. Therefore the Random Forest's strengths are very suitable for classification of data from the welding dataset.


2. **Gradient Boosting**, i.e. `HistGradientBoostingClassifier` from `scikit-learn`
    - The ensemble method of Gradient Boosting has a high predictive accuracy and certain variations work well with large datasets `HistGradientBoostingClassifier` was chosen over `GradientBoostingClassifier` as it has been designed to be scalable and is therefore faster for large datasets like the welding dataset. Additionally it includes regularization parameters to reduce the overfitting tendency of Gradient Boosting. This and its ability to handle high-dimensional dataset makes it suitable for use with the welding dataset.


3. **Support Vector Machines** (SVM), i.e. `RBFSampler ` with `SGDClassifier` from `scikit-learn`
    - SVMs are particularly useful in datasets with a large amont of dimentions or features (20 in this case), especially when the number of features is lower than the amount of samples, which is the case in the welding dataset. In such cases SVMs present a low tendency of overfitting. The multiple Kernel functions available provide room to experiment for finding an optimal model. The ability of SVMs to capture complex feature relationships also makes it suitable for use with the welding dataset.
    - However as full SVMs with non-linear kernels scale badly with large datasetes (as in this case), a combination of `RBFSampler` and `SGDClassifier` is used to approximate the SVM.


4. **Multi-layer Perceptron** (MLP), i.e. `MLPClassifier` from `scikit-learn`
    - MLPs are able to model complex non-linear relationships between features and are suitable for high-dimensional labeled data, as present in the welding dataset. MLP does have a tendency to overfit, especially when the feature relations are very complex and the training dataset is comparatively small. As the welding dataset contains over 300,000 samples, this issue should be somewhat mitigated. Additionally MLPs are very sensitive to tuning of hyperparameters, therefore requiring a high level of care during the selection process. Their high flexibility and good prediction performance when tuned properly still makes MLPs suitable for use with the welding dataset.

## Exercise 5.2: Hyperparameter Search Setup

Regarding class imbalance: for the Random Forest and the SVM classifier class weight was used, as the class imbalance in the dataset is not extreme (roughly 57 : 43). For the other classifiers that do not support the class weight parameter, SMOTE was used.

In [4]:
# split dataset into train and held-out test dataset
train_test_seed = 38

data_train, data_test, labels_train, labels_test = train_test_split(
    data, labels,
    test_size=0.2,
    stratify=labels,
    random_state=train_test_seed
)

# extract smaller dataset for hyperparameter tuning to reduce computational load
data_tune, _, labels_tune, _ = train_test_split(
    data_train, labels_train,
    test_size=0.9,           # Keep 10%
    stratify=labels_train,
    random_state=train_test_seed
)

# extract features from train and test datasets seperately
features_train = extract_features(data_train, labels_train)
features_test = extract_features(data_test, labels_test)
features_tune = extract_features(data_tune, labels_tune)

# remove labels column from extracted features data
features_train.drop('labels', axis=1, inplace=True)
features_test.drop('labels', axis=1, inplace=True)
features_tune.drop('labels', axis=1, inplace=True)

In [5]:
# predefine random seed variable to overwrite later
tuning_seed = 12

# define parameter grids and pipelines per classifier

# Random Forest
pipeline_rf = Pipeline([
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=tuning_seed))
])

param_grid_rf = {
    'clf__n_estimators': [10, 50, 100, 200],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5]
}

# Gradient Boosting
pipeline_hgb = ImbPipeline([
    ('smote', SMOTE(random_state=tuning_seed)),
    ('scaler', StandardScaler()),
    ('clf', HistGradientBoostingClassifier(random_state=tuning_seed))
])

param_grid_hgb = {
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_iter': [100, 200],
    'clf__max_depth': [None, 10],
    'clf__min_samples_leaf': [20, 30]
}

# Support Vector Machine
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('rbf_sampler', RBFSampler(random_state=tuning_seed)),
    ('sgd', SGDClassifier(loss='hinge', learning_rate='optimal', random_state=tuning_seed, max_iter=1000, tol=1e-3))
])

param_grid_svm = {
    'rbf_sampler__gamma': [0.01, 0.1, 1, 10],
    'rbf_sampler__n_components': [500, 1000],
    'sgd__alpha': [1e-4, 1e-3, 1e-2],
}

# Multi-layer Perceptron
pipeline_mlp = ImbPipeline([
    ('smote', SMOTE(random_state=tuning_seed)),
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(max_iter=300, random_state=tuning_seed))
])

param_grid_mlp = {
    'clf__hidden_layer_sizes': [(100,), (50, 50)],
    'clf__activation': ['relu', 'tanh'],
    'clf__alpha': [0.0001, 0.001],
    'clf__learning_rate_init': [0.001, 0.01]
}


# function to call later to fit each model with specifies evaluation metrics

def tune_hyperparameters(pipeline, param_grid, metrics, data, labels):
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=metrics,
        refit='f1_macro',
        cv=5,
        return_train_score=False,
        verbose = 3,
        n_jobs=8
    )
    grid.fit(data, labels)

    return grid

## 5.3. Evaluation Metrics

Selected Metrics:
1. **F1-score**
    - The F1-score is useful when classes are imbalanced (as in the welding dataset, approx. 57:43 ratio) and both false positives as well as false negatives are important. Although a false negative in the welding context is less damaging than a false positive, it still constitutes wasted resources.

2. **ROC AUC**
    - (receiver-operating characteristic curve, area under the curve)
    - ROC AUC is used to display the model's ranking ability, i.e. how well the classifier can separate the classes. It is therefore a useful visual tool to quickly and intuitively compare models of different parameters.

3. **Recall**
    - Recall is especially good at identifying false negatives. As stated before, false negatives are more crucial in a welding context, especially as safety is concerned.

In [8]:
evaluation_metrics = {
    'f1_macro': make_scorer(f1_score, average='macro'),
    'recall_macro': make_scorer(recall_score, average='macro'),
    'roc_auc': 'roc_auc'
}

## 5.4 Model Training and Tuning

### Hyperparameter Tuning:

In [ ]:

classifiers =  {
    'RandomForest' : [pipeline_rf, param_grid_rf],
    'HistGradientBoosting' : [pipeline_hgb, param_grid_hgb],
    'SVM' : [pipeline_svm, param_grid_svm],
    'MLP' : [pipeline_mlp, param_grid_mlp]
}

feature_set = ['raw', 'extracted']
data_tune_flat = data_tune.reshape(data_tune.shape[0], -1)

results = []

for clf in classifiers.keys():
    for feat in feature_set:
        pipeline = classifiers.get(clf)[0]
        
        
        params = classifiers.get(clf)[1]

        curr_data = data_tune_flat if feat == 'raw' else features_tune

        grid = tune_hyperparameters(
            pipeline=pipeline,
            param_grid=params, 
            metrics=evaluation_metrics,
            data=curr_data,
            labels=labels_tune
        )

        print(f"Best params for {clf} - {feat}: {grid.best_params_}")
        results.append({'classifier': clf, 'features': feat, 'best_params' : grid.best_params_})

results_df = pd.DataFrame(results)
results_df.to_csv(result_path/'hyperparameter_tuning_results.csv', index=False)
print("Saved results to hyperparameter_tuning_results.csv")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best params for SVM - raw: {'rbf_sampler__gamma': 0.01, 'rbf_sampler__n_components': 1000, 'sgd__alpha': 0.0001}
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best params for SVM - extracted: {'rbf_sampler__gamma': 0.1, 'rbf_sampler__n_components': 1000, 'sgd__alpha': 0.0001}
Saved results to hyperparameter_tuning_results.csv


**Best Parameters per Classification Algorithm**:

1. Random Forest:
    | Parameter           | raw data      | extracted features |
    | ------------------- | ------------- | ------------------ |
    | `n_estimators`      | 200           | 100                |
    | `max_depth`         | None          | None               |
    | `min_samples_split` | 2             | 5                  |

---

2. Gradient Boosting:
    | Parameter           | raw data      | extracted features |
    | ------------------- | ------------- | ------------------ |
    | `learning_rate`     | 0.1           | 0.1                |
    | `max_iter`          | 200           | 200                |
    | `max_depth`         | None          | None               |
    | `min_samples_leaf`  | 20            | 20                 |

---

3. SVM:
    | Parameter                    | raw data      | extracted features |
    | ---------------------------- | ------------- | ------------------ |
    | `RBFSampler`: `gamma`        | 0.01          | 0.1                |
    | `RBFSampler`: `n_components` | 1000          | 1000               |
    | `SGD`: `alpha`               | 0.0001        | 0.0001              |

---

4. MLP:
    | Parameter            | raw data      | extracted features |
    | -------------------- | ------------- | ------------------ |
    | `hidden_layer_sizes` | (100,)        | (100,)             |
    | `activation`         | 'tanh'        | 'relu'             |
    | `alpha`              | 0.001         | 0.001              |
    | `learning_rate_init` | 0.001         | 0.001              |



### Model training:

In [ ]:
best_params = {
    'Random_Forest_Raw' : [pipeline_rf,
                {'clf__n_estimators': 200, 'clf__max_depth': None, 'clf__min_samples_split': 2,
                 'clf__verbose': 2}],
    'Random_Forest_Extracted' : [pipeline_rf,
                {'clf__n_estimators': 100, 'clf__max_depth': None, 'clf__min_samples_split': 5,
                 'clf__verbose': 2}],
    'Gradient_Boosting_Raw' : [pipeline_hgb,
                {'clf__learning_rate': 0.1, 'clf__max_iter': 200, 'clf__max_depth': None, 'clf__min_samples_leaf': 20,
                 'clf__verbose': 2}],
    'Gradient_Boosting_Extracted' : [pipeline_hgb,
                {'clf__learning_rate': 0.1, 'clf__max_iter': 200, 'clf__max_depth': None, 'clf__min_samples_leaf': 20,
                 'clf__verbose': 2}],
    'SVM_Raw' : [pipeline_svm,
                {'rbf_sampler__gamma': 0.01, 'rbf_sampler__n_components': 1000, 'sgd__alpha': 0.0001,
                 'sgd__verbose': 2}],
    'SVM_Extracted' : [pipeline_svm,
                {'rbf_sampler__gamma': 0.1, 'rbf_sampler__n_components': 1000, 'sgd__alpha': 0.0001,
                 'sgd__verbose': 2}],
    'MLP_Raw' : [pipeline_mlp,
                {'clf__hidden_layer_sizes': (100,), 'clf__activation': 'tanh', 'clf__alpha': 0.001, 'clf__learning_rate_init': 0.001,
                 'clf__verbose': True}],
    'MLP_Extracted' : [pipeline_mlp,
                {'clf__hidden_layer_sizes': (100,), 'clf__activation': 'relu', 'clf__alpha': 0.001, 'clf__learning_rate_init': 0.001,
                 'clf__verbose': True}]
}

evaluation_metrics_func = {
    'f1_macro': lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro'),
    'recall_macro': lambda y_true, y_pred: recall_score(y_true, y_pred, average='macro'),
    'roc_auc': lambda y_true, y_proba: roc_auc_score(
        y_true, 
        y_proba[:, 1] if y_proba.ndim == 2 and y_proba.shape[1] == 2 else y_proba
    )
}

prediction_path = Path("model_analysis/ex_05/label_prediction")
prediction_path.mkdir(parents=True, exist_ok=True)

pd.DataFrame(labels_test).to_csv(prediction_path / f"labels_test_true.csv", index=False)

In [ ]:
def fit_model_best_params(pipeline: Pipeline, best_params, X_train, y_train, seed):
    # update pipeline with best parameters
    pipe = clone(pipeline)
    pipe.set_params(**best_params)

    # update various random states across different models to current seed
    if 'clf__random_state' in pipe.get_params():
        pipe.set_params(clf__random_state=seed)
    
    if 'smote__random_state' in pipe.get_params():
        pipe.set_params(smote__random_state=seed)
    
    if 'rbf_sampler__random_state' in pipe.get_params():
        pipe.set_params(rbf_sampler__random_state=seed)

    if 'sgd__random_state' in pipe.get_params():
        pipe.set_params(sgd__random_state=seed)
    
    # fit model to training dataset
    pipe.fit(X_train, y_train)

    return pipe

def evaluate_model(model_name, pipeline, params, seed):
    start = time.time()
    with open("job_status.log", "a") as f:
        f.write(f"[START] {model_name} | Seed: {seed} | Time: {time.ctime()}\n")

    print(f"Evaluating: {model_name}")

    # select correct dataset based on model name
    if 'Raw' in model_name:
        X_train = data_train_flat
        X_test = data_test_flat
    else:
        X_train = features_train
        X_test = features_test

    result = {'model': model_name, 'seed': seed}

    #for seed in seeds:
    print(f"Processing seed: {seed}")
    model = fit_model_best_params(pipeline=pipeline, best_params=params, X_train=X_train, y_train=labels_train, seed=seed)

    # test model on held-out test dataset
    print("Model trained, starting prediction")
    y_pred = model.predict(X_test)

    pd.DataFrame(y_pred).to_csv(prediction_path / f"{model_name}_seed_{seed}_labels_pred.csv", index=False)

    # Get continuous prediction probabilites for ROC AUC metric
    y_proba = None
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)
    elif hasattr(model, 'decision_function'):
        y_proba = model.decision_function(X_test)

    # Compute quality metrics
    print("Prediction finished, calculating scores")
    for name, func in evaluation_metrics_func.items():
        # special case for ROC AUC calculation, capture errors as NAN values
        try:
            if name == 'roc_auc' and y_proba is not None:
                score = func(labels_test, y_proba)
            else:
                score = func(labels_test, y_pred)
        except Exception as e:
            print(f"{func} error: {e}")
            score = np.nan
        result[name] = score
        print(f"{name}: {score}")
    
    elapsed = time.time() - start
    with open("job_status.log", "a") as f:
        f.write(f"[END]   {model_name} | Seed: {seed} | Duration: {elapsed/60:.2f} min\n")
        f.write(f"RESULT: {result}\n")

    return result


# setup random seeds
seeds = [182, 54, 906]

data_train_flat = data_train.reshape(data_train.shape[0], -1)
data_test_flat = data_test.reshape(data_test.shape[0], -1)


jobs = [
    delayed(evaluate_model)(model_name, pipeline, params, seed)
    for model_name, (pipeline, params) in best_params.items()
    for seed in seeds
]

with tqdm_joblib(tqdm(desc="Model evaluation", total=len(jobs))) as progress_bar:
    seed_results = Parallel(n_jobs=6, verbose=1)(jobs)

df = pd.DataFrame(seed_results)
summary_df = df.groupby('model').agg({k: ['mean', 'std'] for k in evaluation_metrics_func}).reset_index()
summary_df.columns = ['model'] + [f'{k}_{stat}' for k in evaluation_metrics_func for stat in ['mean', 'std']]

summary_df.to_csv(result_path/'model_evaluation_metrics.csv', index=False)
print("Saved evaluation results to model_evaluation_metrics.csv")


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.























[Parallel(n_jobs=6)]: Done  24 out of  24 | elapsed: 28.1min finished
100%|██████████| 24/24 [28:04<00:00, 70.17s/it]

Saved evaluation results to model_evaluation_metrics.csv
